# AviSense — train the bird detector on Google Colab

Trains the AirBirds bird detector on Colab's GPU so the ~135 GB of training data never
has to live on your laptop. The data is downloaded straight from the authors' Hugging
Face mirror, cut into 640 px tiles one chunk at a time (each chunk is deleted after
tiling, so Colab's disk never fills up), and the model is trained and tested. Only the
trained model (~6 MB) and the run's plots/logs are kept, in your Google Drive.

Default setup = the current best model in the README: **YOLO11n with a P2 head,
motion-stacked input (frames t−4, t−2, t), hard negatives, AirBirds chunks 0, 5, 6, 7 +
the D02 2021-07-21 sequence**; tested on the D02 2021-06-28 sequence.

**Before you start**
1. *Runtime → Change runtime type → T4 GPU* (or better).
2. In Google Drive create the folder **`MyDrive/AviSense`** and upload into it:
   - **`avisense_code.zip`** — the project code (on the laptop: `outputs/avisense_code.zip`).
   - *optional* **`hard_negative_model.pt`** — a previous model used to find false
     positives to train on (e.g. a copy of `models/airbirds_tiles_temporal2_hn_yolo11n-p2_640.pt`).
     Without it, training uses only random empty tiles (slightly more false positives).
3. *Runtime → Run all*.

**Time (T4):** building the tiles ~1–1.5 h, training ~3–5 h, testing ~20 min. Free Colab
can disconnect long sessions: checkpoints are saved to Drive, and section 6 resumes
training. To skip rebuilding the tiles after a disconnect, set `SAVE_TILES_TO_DRIVE = True`
(needs ~6 GB of free Drive space).

**Result:** `MyDrive/AviSense/models/<RUN_NAME>.pt` → download it into the laptop's
`models/` folder. The dashboard lists it automatically; use image size 1920.

In [ ]:
# ---- Settings ----
DRIVE_DIR = "/content/drive/MyDrive/AviSense"
CODE_ZIP = f"{DRIVE_DIR}/avisense_code.zip"
GIT_URL = ""                       # alternatively, a GitHub URL to clone instead of the zip

RUN_NAME = "airbirds_tiles_temporal2_hn_yolo11n-p2_640_colab"  # keep 'temporal<k>' in the name for motion models
MODEL_CFG = "configs/models/yolo11n-p2.yaml"   # or "yolo11n.pt" for the plain model
TEMPORAL = 2                       # motion input: frames t-2k, t-k, t; 0 = ordinary colour frames
TRAIN_EVERY = 2                    # use every 2nd training frame (consecutive frames are near-duplicates)
HARD_NEG_WEIGHTS = f"{DRIVE_DIR}/hard_negative_model.pt"  # optional; skipped if the file is missing

TRAIN_CHUNKS = ["0", "5", "6", "7", "test_D02_20210721142744"]
VAL_CHUNK = "9"
HELD_OUT = "D02_20210728064810"    # D02 sequence moved from train (chunk 7) to val
TEST_SEQ = "test_D02_20210628090856"

EPOCHS, PATIENCE, BATCH, WORKERS = 80, 25, 16, 2
SAVE_TILES_TO_DRIVE = False        # True: keep a ~6 GB zip of the tiles in Drive for resuming

## 1. Setup

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
Path(DRIVE_DIR, "models").mkdir(parents=True, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /content | tail -1

In [ ]:
REPO = Path("/content/AviSense")
if not REPO.exists():
    if GIT_URL:
        !git clone -q {GIT_URL} {REPO}
    else:
        !unzip -q -o {CODE_ZIP} -d {REPO}
os.chdir(REPO)
!pip install -q "ultralytics>=8.3,<9" lap
print("Code ready:", sorted(p.name for p in REPO.iterdir()))

## 2. Data helpers
Each AirBirds chunk is a 12–22 GB zip. `fetch()` downloads one (resuming if the
connection drops), extracts it flat into `/content/raw/{images,labels}/<split>` and
deletes the zip.

In [ ]:
HF = "https://huggingface.co/datasets/auniquesun/AirBirds/resolve/main"
RAW, DL, TILES = Path("/content/raw"), Path("/content/dl"), Path("/content/tiles")
for d in (RAW, DL, TILES):
    d.mkdir(parents=True, exist_ok=True)

def run(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)

def fetch(chunk, split):
    for kind in ("images", "labels"):
        name = f"{kind}{chunk}.zip" if chunk.isdigit() else f"{kind}_{chunk}.zip"
        z = DL / name
        run(f"curl -sSL --retry 20 --retry-all-errors -C - -o {z} {HF}/{name}")
        (RAW / kind / split).mkdir(parents=True, exist_ok=True)
        run(f"unzip -q -o -j {z} -d {RAW / kind / split}")
        z.unlink()
    n = len(list((RAW / "images" / split).glob("*.png")))
    print(f"chunk {chunk}: {n} {split} frames on disk")

def clear(split):
    for kind in ("images", "labels"):
        shutil.rmtree(RAW / kind / split, ignore_errors=True)

def make_tiles(split, extra=""):
    hn = f"--hard-negatives {HARD_NEG_WEIGHTS}" if split == "train" and Path(HARD_NEG_WEIGHTS).exists() else ""
    run(f"{sys.executable} -m src.detection.make_tiles --src {RAW} --out {TILES} --splits {split} --append "
        f"--temporal {TEMPORAL} --train-every {TRAIN_EVERY} --workers {WORKERS} {hn} {extra}")

print("hard negatives:", "yes" if Path(HARD_NEG_WEIGHTS).exists() else "no (file not found)")

## 3. Build the training and validation tiles
One chunk at a time: download → extract → tile → delete. The held-out D02 sequence is
moved to the validation set so model selection also sees the test camera.

In [ ]:
TILES_ZIP = Path(DRIVE_DIR, f"tiles_temporal{TEMPORAL}.zip")
if TILES_ZIP.exists():
    run(f"unzip -q -o {TILES_ZIP} -d {TILES}")
    print("Restored tiles from Drive")
else:
    for chunk in TRAIN_CHUNKS:
        fetch(chunk, "train")
        held = list((RAW / "images" / "train").glob(f"{HELD_OUT}_*.png"))
        if held:  # move the held-out sequence to val before tiling
            for kind, ext in (("images", "png"), ("labels", "txt")):
                (RAW / kind / "val").mkdir(parents=True, exist_ok=True)
                for f in (RAW / kind / "train").glob(f"{HELD_OUT}_*.{ext}"):
                    shutil.move(str(f), RAW / kind / "val" / f.name)
            print(f"moved {len(held)} {HELD_OUT} frames to val")
        make_tiles("train")
        clear("train")
        !df -h /content | tail -1
    fetch(VAL_CHUNK, "val")
    make_tiles("val")
    clear("val")
    if SAVE_TILES_TO_DRIVE:
        run(f"cd {TILES} && zip -q -r -0 {TILES_ZIP} images labels")

for split in ("train", "val"):
    print(split, len(list((TILES / "images" / split).glob("*.png"))), "tiles")

## 4. Train

In [ ]:
# 'temporal<k>' in the dataset file name switches off hue/saturation augmentation (see train.py).
DATA_YAML = REPO / "configs" / (f"airbirds_tiles_temporal{TEMPORAL}_colab.yaml" if TEMPORAL else "airbirds_tiles_colab.yaml")
DATA_YAML.write_text(f"path: {TILES}\ntrain: images/train\nval: images/val\nnames:\n  0: bird\n")
RUNS = f"{DRIVE_DIR}/runs"
pretrained = "--pretrained yolo11n.pt" if MODEL_CFG.endswith(".yaml") else ""

TRAIN_CMD = (f"{sys.executable} -m src.detection.train --model {MODEL_CFG} {pretrained} --data {DATA_YAML} "
             f"--imgsz 640 --batch {BATCH} --workers {WORKERS} --epochs {EPOCHS} --patience {PATIENCE} "
             f"--name {RUN_NAME} --project {RUNS} --no-test")
!{TRAIN_CMD}
shutil.copy(REPO / "models" / f"{RUN_NAME}.pt", Path(DRIVE_DIR, "models", f"{RUN_NAME}.pt"))
print("Saved", Path(DRIVE_DIR, "models", f"{RUN_NAME}.pt"))

## 5. Test on the D02 2021-06-28 sequence
Full 1920×1080 frames at image size 1920, the way the pipeline runs the model.

In [ ]:
fetch(TEST_SEQ, "test")
if TEMPORAL:  # motion-stacked copies of the test frames, built exactly as the live pipeline sees them
    TEST_ROOT = Path(f"/content/test_temporal{TEMPORAL}")
    run(f"{sys.executable} -m src.detection.temporal --images {RAW}/images/test --labels {RAW}/labels/test "
        f"--step {TEMPORAL} --out {TEST_ROOT}")
    clear("test")
else:
    TEST_ROOT = RAW
TEST_YAML = REPO / "configs" / "airbirds_test_colab.yaml"
TEST_YAML.write_text(f"path: {TEST_ROOT}\ntrain: images/test\nval: images/test\ntest: images/test\nnames:\n  0: bird\n")

TEST_CMD = (f"{sys.executable} -m src.detection.train --evaluate {DRIVE_DIR}/models/{RUN_NAME}.pt "
            f"--test-data {TEST_YAML} --imgsz 1920 --project {RUNS}")
!{TEST_CMD} 2>&1 | tee "{DRIVE_DIR}/models/{RUN_NAME}_test.txt"

## 6. Resume after a disconnect
Run the **Settings** cell and section 1–2, then this cell. It rebuilds the tiles if they
are gone (or restores them from Drive with `SAVE_TILES_TO_DRIVE`), then continues
training from the last checkpoint in `MyDrive/AviSense/runs/<RUN_NAME>`. Afterwards run
section 5.

In [ ]:
train_tiles = TILES / "images" / "train"
if not (train_tiles.exists() and any(train_tiles.glob("*.png"))):
    raise SystemExit("Tiles are missing: run section 3 first (it restores them from Drive if saved).")
!{sys.executable} -m src.detection.train --resume --name {RUN_NAME} --project {DRIVE_DIR}/runs --no-test
shutil.copy(REPO / "models" / f"{RUN_NAME}.pt", Path(DRIVE_DIR, "models", f"{RUN_NAME}.pt"))

## 7. Download the model
The model is in Google Drive at `MyDrive/AviSense/models/<RUN_NAME>.pt` (test results in
`<RUN_NAME>_test.txt`). Download it from drive.google.com, or run the cell below, and put
it in the laptop's `models/` folder.

In [ ]:
from google.colab import files
files.download(f"{DRIVE_DIR}/models/{RUN_NAME}.pt")